In [1]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, dataset

import os
import numpy as np

import pathlib

from PIL import Image
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

import torch.nn.functional as F

In [2]:
import xml.etree.ElementTree as ET
tree = ET.parse("oxford-iiit-pet/annotations/xmls/yorkshire_terrier_180.xml")
root = tree.getroot()

# Print root tag
print("Root:", root.tag)
size = root.find("size")
width = int(size.find("width").text)
height = int(size.find("height").text)

print("Width:", width)
print("Height:", height)

obj = root.find("object")
name = obj.find("name").text
pose = obj.find("pose").text

bbox = obj.find("bndbox")
xmin = int(bbox.find("xmin").text)
ymin = int(bbox.find("ymin").text)
xmax = int(bbox.find("xmax").text)
ymax = int(bbox.find("ymax").text)

print("\nObject:", name)
print("Pose:", pose)
print("Bounding Box:", xmin, ymin, xmax, ymax)

Root: annotation
Width: 375
Height: 500

Object: dog
Pose: Frontal
Bounding Box: 101 102 326 231


In [ ]:
class OxfordIIITPetDataset(Dataset):
    def __init__(self, root_dir, transforms=None):
        self.root = pathlib.Path(root_dir)
        self.images_dir = self.root / "images"
        self.xml_dir = self.root / "annotations" / "xmls"
        self.trimap_dir = self.root / "annotations" / "trimaps"
        self.transforms = transforms

        self.class_map = {}

        # Read list.txt
        self.samples = []
        with open(self.root / "annotations" / "list.txt") as f:
            for line in f:
                if line.startswith("#"):
                    continue
                name, class_id, species, breed_id = line.strip().split()
                class_id = int(class_id)

                # Store mapping (only once per class_id)
                breed_name = "_".join(name.split("_")[:-1])

                if class_id not in self.class_map:
                    self.class_map[class_id] = breed_name

                img_path = self.images_dir / f"{name}.jpg"
                xml_path = self.xml_dir / f"{name}.xml"
                trimap_path = self.trimap_dir / f"{name}.png"

                if img_path.exists() and xml_path.exists() and trimap_path.exists():
                        self.samples.append({
                            "name": name,
                            "class_id": class_id,
                            "species": int(species),
                            "breed_id": int(breed_id)
                        })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        name = sample["name"]

        # Paths
        img_path = self.images_dir / f"{name}.jpg"
        xml_path = self.xml_dir / f"{name}.xml"
        trimap_path = self.trimap_dir / f"{name}.png"

        # Load image
        image = Image.open(img_path).convert("RGB")
        image = torch.from_numpy(np.array(image)).permute(2, 0, 1)/255.0
        image = F.interpolate(image.unsqueeze(0), [224, 224], mode='bilinear')

        # Load mask
        mask = Image.open(trimap_path)
        mask = torch.torch.from_numpy(np.array(mask))/255.0
        mask = F.interpolate(mask.unsqueeze(0).unsqueeze(0), [224,224], mode='bilinear')[0]

        # Parse XML
        tree = ET.parse(xml_path)
        root = tree.getroot()

        bbox = root.find("object").find("bndbox")
        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)

        if self.transforms:
            image = self.transforms(image)

        return {
            "image": image[0],
            "name": name,
            "bbox": [xmin, ymin, xmax, ymax],
            "mask": mask[0],
            "species": sample["species"],   # 1=cat, 2=dog
            "breed": sample["breed_id"],
            "class_id": sample["class_id"],
            "breed_name": self.class_map[sample["class_id"]]
        }
    
dataset = OxfordIIITPetDataset(root_dir = "oxford-iiit-pet")

In [13]:
batch = next(iter(DataLoader(dataset, batch_size = 2, shuffle=True)))

plt.imshow(batch['image'].squeeze(0).permute(1, 2, 0))
plt.title(batch['breed_name'])
print(batch['name'])
print(batch["class_id"])

> /var/folders/3p/xlr6tgyx4t980qpxnnrs12kc0000gn/T/ipykernel_9970/444888213.py(60)__getitem__()
     58         import pdb;
     59         pdb.set_trace()
---> 60         mask = F.interpolate(mask, [224, 224], mode='bilinear')
     61 
     62         # Parse XML

torch.Size([500, 375])
*** ValueError: Input and output must have the same number of spatial dimensions, but got input with spatial dimensions of [3, 224, 224] and output size of [224, 224]. Please provide input tensor in (N, C, d1, d2, ...,dK) format and output size in (o1, o2, ...,oK) format.
tensor([[[[1.7923e-01, 1.6079e-01, 1.1865e-01,  ..., 1.5252e-01,
           1.9036e-01, 2.1314e-01],
          [1.7525e-01, 1.2548e-01, 1.1961e-01,  ..., 2.4613e-01,
           2.1747e-01, 2.3694e-01],
          [1.1248e-01, 9.7906e-02, 1.3617e-01,  ..., 2.3023e-01,
           2.1553e-01, 2.3619e-01],
          ...,
          [2.6465e-02, 3.0168e-02, 1.3019e-02,  ..., 4.5320e-01,
           3.5051e-01, 3.8201e-01],
          [1.9213e-

In [6]:
batch['image'].shape

torch.Size([1, 3, 224, 224])